## **Waffle Chart**

In [ ]:
# =========================
# 1. IMPORTS
# =========================
from altair.datasets import data
import altair as alt
import pandas as pd

# =========================
# 2. CARGA Y LIMPIEZA
# =========================
movies = data.movies()

# Filtrar películas que NO recuperaron presupuesto
filtered_data = movies[movies['Worldwide Gross'] <= movies['Production Budget']].copy()

# Crear métricas
filtered_data['Porcentaje Recaudación'] = (
    filtered_data['Worldwide Gross'] / filtered_data['Production Budget']
)

# Rellenar valores faltantes
filtered_data['IMDB Rating'] = filtered_data['IMDB Rating'].fillna(
    filtered_data['IMDB Rating'].mean()
)

# Crear bins de IMDB
filtered_data['IMDB Rating Bins'] = pd.cut(
    filtered_data['IMDB Rating'],
    bins=[0, 4, 5, 6, 7, 8, 10],
    labels=['0-4', '4-5', '5-6', '6-7', '7-8', '8-10']
)

# =========================
# 3. AGRUPACIÓN
# =========================
grouped_data = (
    filtered_data
    .groupby('IMDB Rating Bins')[['Worldwide Gross', 'Production Budget']]
    .sum()
    .reset_index()
)

grouped_data['Porcentaje Recaudación'] = (
    grouped_data['Worldwide Gross'] / grouped_data['Production Budget']
)

# =========================
# 4. GENERAR WAFFLE DATA
# =========================
n_squares = 100
waffle_data = []

for _, row in grouped_data.iterrows():
    filled = min(100, int(row["Porcentaje Recaudación"] * n_squares))

    for i in range(n_squares):
        waffle_data.append({
            "IMDB Rating Bins": row["IMDB Rating Bins"],
            "x": i % 10,
            "y": i // 10,
            "filled": i < filled
        })

waffle_df = pd.DataFrame(waffle_data)

# =========================
# 5. VARIABLES PARA TOOLTIP
# =========================
# Mapear porcentaje por grupo
percent_map = dict(zip(
    grouped_data['IMDB Rating Bins'],
    grouped_data['Porcentaje Recaudación']
))

waffle_df['Porcentaje'] = waffle_df['IMDB Rating Bins'].map(percent_map)

# Formato %
waffle_df['Porcentaje_label'] = (
    (waffle_df['Porcentaje'] * 100)
    .round(1)
    .astype(str) + '%'
)

# Estado visual
waffle_df['Estado'] = waffle_df['filled'].map({
    True: 'Recaudado',
    False: 'No recaudado'
})

# =========================
# 6. VISUALIZACIÓN
# =========================
chart = alt.Chart(waffle_df).mark_square(size=100).encode(
    x=alt.X("x:O", axis=None),
    y=alt.Y("y:O", axis=None),
    color=alt.Color(
        "Estado:N",
        scale=alt.Scale(
            domain=['Recaudado', 'No recaudado'],
            range=["#1f77b4", "#e0e0e0"]
        ),
        legend=alt.Legend(title="Interpretación")
    ),
    tooltip=[
        alt.Tooltip("IMDB Rating Bins:N", title="Rango IMDB"),
        alt.Tooltip("Estado:N", title="Estado"),
        alt.Tooltip("Porcentaje_label:N", title="% Recaudado")
    ]
).facet(
    facet='IMDB Rating Bins:N',
    columns=2
).properties(
    title="Relación recaudación / presupuesto por rango de IMDB"
)

chart.save("waffle_chart.html")

c:\Users\esteb\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas\core\dtypes\cast.py:1056: RuntimeWarning: invalid value encountered in cast
  if (arr.astype(int) == arr).all():
c:\Users\esteb\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas\core\dtypes\cast.py:1080: RuntimeWarning: invalid value encountered in cast
  if (arr.astype(int) == arr).all():
C:\Users\esteb\AppData\Local\Temp\ipykernel_6764\2161785282.py:38: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby('IMDB Rating Bins')[['Worldwide Gross', 'Production Budget']]


## **Beeswarm Chart**

In [16]:
# =========================
# 1. IMPORTS
# =========================
import altair as alt
from altair.datasets import data
import pandas as pd

# =========================
# 2. CARGA Y LIMPIEZA
# =========================
movies = data.movies()

# Filtrar datos relevantes
movies = movies.dropna(subset=['IMDB Rating', 'Major Genre'])

# Opcional: limitar géneros con suficientes datos (mejora visual)
top_genres = movies['Major Genre'].value_counts().nlargest(10).index
movies = movies[movies['Major Genre'].isin(top_genres)]

# =========================
# 3. BEESWARM (mejorado)
# =========================
beeswarm = alt.Chart(movies).transform_calculate(
    # Jitter más natural (efecto enjambre)
    jitter='(random() - 0.5) * 0.6'
).mark_circle(
    size=45,
    opacity=0.65
).encode(
    x=alt.X(
        'IMDB Rating:Q',
        title='Puntuación IMDB',
        scale=alt.Scale(domain=[0, 10])
    ),
    y=alt.Y(
        'Major Genre:N',
        title='Género'
    ),
    yOffset='jitter:Q',
    color=alt.Color(
        'Major Genre:N',
        legend=alt.Legend(title="Género"),
        scale=alt.Scale(scheme='tableau10')
    ),
    tooltip=[
        alt.Tooltip('Title:N', title='Película'),
        alt.Tooltip('IMDB Rating:Q', title='Rating'),
        alt.Tooltip('Major Genre:N', title='Género')
    ]
).properties(
    width=700,
    height=400,
    title='Distribución de puntuaciones IMDB por género (Beeswarm)'
)

beeswarm.save('beeswarm.html')

c:\Users\esteb\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas\core\dtypes\cast.py:1056: RuntimeWarning: invalid value encountered in cast
  if (arr.astype(int) == arr).all():
c:\Users\esteb\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas\core\dtypes\cast.py:1080: RuntimeWarning: invalid value encountered in cast
  if (arr.astype(int) == arr).all():


## **Choropleth Chart**

In [17]:
import altair as alt
from altair.datasets import data
import pandas as pd
import pycountry

# Función para convertir país → ISO numérico
def get_iso_numeric(country_name):
    try:
        return int(pycountry.countries.lookup(country_name).numeric)
    except:
        return None

# Dataset
gapminder = data.gapminder()
df = gapminder[gapminder['year'] == 2005].copy()

# Convertir a ISO
df['id'] = df['country'].apply(get_iso_numeric)

# Mapa
world = alt.topo_feature(data.world_110m.url, 'countries')

# Choropleth
choropleth = alt.Chart(world).mark_geoshape().encode(
    color=alt.Color('life_expect:Q', scale=alt.Scale(scheme='blues')),
    tooltip=['country:N', 'life_expect:Q']
).transform_lookup(
    lookup='id',
    from_=alt.LookupData(df, 'id', ['country', 'life_expect'])
).project('naturalEarth1').properties(
    title='Esperanza de vida por país (ISO mapping)'
)

choropleth

alt.Chart(...)